In [1]:
import pandas as pd
import numpy as np

def read_excel(file_name):
    df = pd.read_excel(file_name)
    return df

def read_txt(file_name):
    file = open(file_name)
    lines = file.readlines()
    return(lines[0])

In [2]:
import os
import glob

def get_files(subfolder, extension):
    dir = f"{os.getcwd()}/content/{subfolder}/"
    tables = glob.glob(f"{dir}*.{extension}")
    return tables

In [3]:
class Analizer:
    def __init__(self, boundary):
        self.results = get_files(subfolder="results", extension="xlsx")
        self.results_df = pd.DataFrame()
        self.boundary = boundary
    
    def has_minimum_requirements(self, df, sort_by="r2"):
        sorted_df = df.sort_values(by=sort_by, ascending=False)
        top_r2 = sorted_df.head(1)[sort_by].values[0]
        if top_r2 < self.boundary:
            return False
        return True
    
    def concatenate_df(self, df, architecture):
        if self.has_minimum_requirements(df):
            df['Architecture'] = architecture
            df = df.rename(columns={'Unnamed: 0': 'model'})
            self.results_df = pd.concat([self.results_df, df], ignore_index=True) 

    def create_results_df(self):
        for file in self.results:
            df = read_excel(file)
            architecture = read_txt(file.replace(".xlsx", ".txt"))
            self.concatenate_df(df, architecture)
        self.results_df = self.results_df.sort_values(by="r2", ascending=False, ignore_index=True)

    def discard_below_average(self, sort_by):
        column_mean = self.results_df[sort_by].mean()      
        self.results_df = self.results_df[self.results_df[sort_by] >= column_mean]
    
    def discard_high_standard_deviation(self):
        r2_val, r2_test = self.results_df['r2_val'], self.results_df['r2_test']
        std_devs = np.abs(r2_val - r2_test)
        mean_std_dev = std_devs.mean()
        self.results_df = self.results_df[std_devs < mean_std_dev]

    def clean_folder(self, subfolder, extension, remove_last=True):
        files = get_files(subfolder, extension)
        models = self.results_df["model"]
        if (remove_last):
            models = models.apply(lambda x: '_'.join(x.rsplit('_', 1)[:-1]))
        for file in files:
            file_name = os.path.basename(file).split('.')[0]
            file_parts = file_name.split('_')            
            dataset_model = f"model_{file_parts[1]}_{file_parts[2]}" 
            if (remove_last == False):
                dataset_model = (f"{dataset_model}_{file_parts[3]}")
            if dataset_model not in models.values:
                os.remove(file)   
        
    def Analize(self):
        self.create_results_df()
        self.discard_below_average(sort_by="r2")
        self.discard_below_average(sort_by="r2_vt")
        self.discard_high_standard_deviation()
        self.results_df.to_excel(f"better_results.xlsx", index=True)
        display(self.results_df)


In [4]:
analize = Analizer(0.9)
analize.Analize()
analize.clean_folder(subfolder="dataset", extension="pkl")
analize.clean_folder(subfolder="results", extension="xlsx")
analize.clean_folder(subfolder="results", extension="txt")
analize.clean_folder(subfolder="models", extension="keras", remove_last=False)



,model,r2,r2_sup,r2_test,r2_val,r2_vt,mse,mse_sup,mse_test,mse_val,mse_vt,mape,rmse,r2_adj,rsd,aic,bic,Architecture
0,model_25_9_17,0.999398,0.674411,0.981573,0.997647,0.992228,0.004028,2.177216,0.020294,0.003600,0.011947,0.106670,0.063465,1.000127,0.066167,287.029040,455.233904,"Hidden Size=[9, 10], regularizer=0.5, learning..."
1,model_25_9_16,0.999397,0.674428,0.981711,0.997734,0.992320,0.004030,2.177103,0.020142,0.003466,0.011804,0.105382,0.063485,1.000127,0.066188,287.027807,455.232670,"Hidden Size=[9, 10], regularizer=0.5, learning..."
2,model_25_9_18,0.999397,0.674395,0.981447,0.997567,0.992142,0.004031,2.177323,0.020433,0.003723,0.012078,0.107829,0.063488,1.000127,0.066191,287.027596,455.232459,"Hidden Size=[9, 10], regularizer=0.5, learning..."
3,model_25_9_19,0.999396,0.674380,0.981332,0.997494,0.992065,0.004037,2.177423,0.020559,0.003835,0.012197,0.108875,0.063541,1.000127,0.066246,287.024292,455.229156,"Hidden Size=[9, 10], regularizer=0.5, learning..."
4,model_25_9_15,0.999396,0.674446,0.981861,0.997829,0.992421,0.004040,2.176984,0.019977,0.003321,0.011649,0.103952,0.063561,1.000127,0.066267,287.023036,455.227900,"Hidden Size=[9, 10], regularizer=0.5, learning..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1468,model_19_2_0,0.968653,0.752940,0.939417,0.988196,0.986291,0.209621,1.652091,0.045931,0.186262,0.116097,0.534244,0.457844,1.008267,0.477335,233.124910,373.295630,"Hidden Size=[10, 7], regularizer=0.3, learning..."
1480,model_19_5_3,0.968267,0.769862,0.966813,0.997067,0.995789,0.212199,1.538932,0.037521,0.026039,0.031780,0.438103,0.460650,1.008369,0.480261,233.100463,373.271183,"Hidden Size=[10, 7], regularizer=0.3, learning..."
1483,model_27_5_3,0.968224,0.711363,0.972989,0.974005,0.973613,0.212487,1.930120,0.097740,0.116421,0.107081,1.553799,0.460964,1.006690,0.480588,279.097745,447.302609,"Hidden Size=[9, 10], regularizer=0.3, learning..."
1489,model_21_4_4,0.967813,0.662173,0.941034,0.982176,0.972653,0.215232,2.259048,0.154982,0.147363,0.151173,1.091165,0.463931,1.006717,0.483681,281.072079,450.495819,"Hidden Size=[6, 15], regularizer=0.5, learning..."
